## 1. Khởi tạo Spark

In [1]:
import os, sys

os.environ["SPARK_LOCAL_IP"] = "127.0.0.1"
os.environ["PYSPARK_PYTHON"] = sys.executable
# Spark trên Windows cần hadoop.dll trong PATH để ghi file
os.environ["PATH"] = os.path.join(os.environ.get("HADOOP_HOME", r"C:\hadoop"), "bin") + os.pathsep + os.environ["PATH"]

import importlib.util
import numpy as np
import pandas as pd
import shapely
from cartopy.io import shapereader
from pyspark.sql import SparkSession, functions as F
from pyspark.sql.types import StructType, StructField, StringType
from pyspark.ml.feature import VectorAssembler
from pyspark.ml.stat import Correlation

spark = (SparkSession.builder
         .master("local[*]")
         .appName("CentreDistanceTmp")
         .config("spark.driver.memory", "8g")
         .config("spark.sql.shuffle.partitions", "16")
         .getOrCreate())
spark.version

'3.5.3'

## 2. Đọc dữ liệu và loại ngoại lai của `price`

Cắt theo nhóm tại điểm gãy trên biểu đồ phân vị: Moscow cắt 0,2% ở đầu dưới và 0,1% ở đầu trên, Saint Petersburg và các vùng còn lại cắt 0,1% mỗi đầu (ngưỡng tính trên toàn bộ dữ liệu vì đây là thử nghiệm).

In [2]:
df = (spark.read.parquet("../data/clean.parquet")
      .withColumn("price_group", F.when(F.col("province").isin("Moscow", "Saint Petersburg"), F.col("province"))
                                  .otherwise("Phần còn lại")))

CUTS = {"Moscow": (0.002, 0.001), "Saint Petersburg": (0.001, 0.001), "Phần còn lại": (0.001, 0.001)}  # (dưới, trên)
levels = sorted({round(p, 6) for lo, hi in CUTS.values() for p in (lo, 1 - hi)})
q = {r["price_group"]: dict(zip(levels, r["q"]))
     for r in df.groupBy("price_group").agg(F.percentile("price", levels).alias("q")).collect()}
bounds = {g: (q[g][round(lo, 6)], q[g][round(1 - hi, 6)]) for g, (lo, hi) in CUTS.items()}

def by_group(values):
    """Biểu thức Spark trả về values[nhóm] theo cột price_group."""
    expr = None
    for g, v in values.items():
        expr = (F.when if expr is None else expr.when)(F.col("price_group") == g, F.lit(v))
    return expr

n_before = df.groupBy("price_group").count().toPandas().set_index("price_group")["count"]
df = df.filter(F.col("price").between(by_group({g: b[0] for g, b in bounds.items()}),
                                      by_group({g: b[1] for g, b in bounds.items()})))
n_after = df.groupBy("price_group").count().toPandas().set_index("price_group")["count"]
cut = pd.DataFrame({"lower": {g: b[0] for g, b in bounds.items()}, "upper": {g: b[1] for g, b in bounds.items()},
                    "before": n_before, "removed": n_before - n_after})
print(cut.to_string(float_format="{:,.0f}".format))
print(f"Tổng loại: {cut['removed'].sum():,} / {n_before.sum():,} bản ghi")

                     lower       upper   before  removed
Moscow           1,150,000 241,480,345   438906     1294
Saint Petersburg 1,100,000 120,000,000   461303      859
Phần còn lại       200,000  26,000,000  4570332     8794
Tổng loại: 10,947 / 5,470,541 bản ghi


Hàm khoảng cách haversine (km) lấy từ `geo_utils.py`.

In [3]:
# Cho phép import các module .py ở thư mục gốc project
sys.path.insert(0, os.path.abspath(".."))
from geo_utils import haversine_km

## 3. Khoảng cách tới thủ phủ của vùng

Thủ phủ lấy từ Natural Earth (`Admin-0/Admin-1 capital`), gán vào vùng bằng ranh giới `admin_1` như `region_lookup.py`; vùng có thêm thủ phủ cũ của khu tự trị đã sáp nhập thì giữ thành phố đông dân nhất.

In [4]:
spec = importlib.util.spec_from_file_location("region_lookup", "../region_lookup.py")
region_lookup = importlib.util.module_from_spec(spec)
spec.loader.exec_module(region_lookup)

places = shapereader.Reader(shapereader.natural_earth("10m", "cultural", "populated_places")).records()
caps = pd.DataFrame([(p.attributes["NAMEASCII"], p.attributes["LATITUDE"], p.attributes["LONGITUDE"], p.attributes["POP_MAX"])
                     for p in places if p.attributes["ADM0_A3"] == "RUS" and "capital" in p.attributes["FEATURECLA"]],
                    columns=["city", "lat", "lon", "pop"])
prov = region_lookup.load_provinces()
ci, pi = shapely.STRtree([g for _, g in prov]).query(shapely.points(caps["lon"], caps["lat"]), predicate="within")
caps["province"] = None
caps.loc[caps.index[ci], "province"] = [prov[i][0] for i in pi]
centres = (caps.dropna(subset=["province"]).sort_values("pop", ascending=False)
               .drop_duplicates("province").set_index("province")[["city", "lat", "lon"]])

Chỉnh sửa: Abakan rơi sang Krasnoyarsk Krai do ranh giới thô nên gán lại cho Khakassia; các thành phố lớn dùng tọa độ trên bài Wikipedia tiếng Anh của thành phố (truy cập 26/10/2026); tỉnh Moscow và tỉnh Leningrad lấy trung tâm là Moscow và Saint Petersburg.

In [5]:
abakan = caps[caps["city"] == "Abakan"].iloc[0]
centres.loc["Republic of Khakassia"] = ["Abakan", abakan["lat"], abakan["lon"]]

WIKI_CENTRES = {
    "Moscow":                    ("Moscow", 55.75583333, 37.61777778),
    "Moscow Oblast":             ("Moscow", 55.75583333, 37.61777778),
    "Saint Petersburg":          ("Saint Petersburg", 59.9375, 30.30861111),
    "Leningrad Oblast":          ("Saint Petersburg", 59.9375, 30.30861111),
    "Novosibirsk Oblast":        ("Novosibirsk", 55.05, 82.95),
    "Krasnodar Krai":            ("Krasnodar", 45.03333333, 38.96666667),
    "Sverdlovsk Oblast":         ("Yekaterinburg", 56.83555556, 60.61277778),
    "Republic of Tatarstan":     ("Kazan", 55.79638889, 49.10888889),
    "Rostov Oblast":             ("Rostov-on-Don", 47.2225, 39.71),
    "Chelyabinsk Oblast":        ("Chelyabinsk", 55.15472222, 61.37583333),
    "Tyumen Oblast":             ("Tyumen", 57.15, 65.53333333),
    "Yaroslavl Oblast":          ("Yaroslavl", 57.6264, 39.8933),
    "Nizhny Novgorod Oblast":    ("Nizhny Novgorod", 56.32694444, 44.0075),
    "Republic of Bashkortostan": ("Ufa", 54.72611111, 55.9475),
    "Krasnoyarsk Krai":          ("Krasnoyarsk", 56.00888889, 92.87194444),
    "Samara Oblast":             ("Samara", 53.20277778, 50.14083333),
    "Altai Krai":                ("Barnaul", 53.34861111, 83.77638889),
    "Voronezh Oblast":           ("Voronezh", 51.67166667, 39.21055556),
    "Stavropol Krai":            ("Stavropol", 45.05, 41.98333333),
    "Chuvash Republic":          ("Cheboksary", 56.15, 47.23333333),
    "Udmurt Republic":           ("Izhevsk", 56.83333333, 53.18333333),
    "Belgorod Oblast":           ("Belgorod", 50.6, 36.6),
    "Perm Krai":                 ("Perm", 58.0, 56.31666667),
    "Kaliningrad Oblast":        ("Kaliningrad", 54.70027778, 20.45305556),
}
for p, v in WIKI_CENTRES.items():
    centres.loc[p] = list(v)

provinces = set(pd.read_csv("../data/raw/region_names.csv")["province"])
print(f"Vùng có trung tâm: {len(provinces & set(centres.index))}/{len(provinces)} | thiếu: {sorted(provinces - set(centres.index))}")

Vùng có trung tâm: 84/84 | thiếu: []


Ghép tọa độ thủ phủ vào từng bản ghi và tính `dist_to_capital` (km).

In [6]:
centres = centres.loc[sorted(provinces)]
cap_lat = F.create_map(*[F.lit(x) for p, r in centres.iterrows() for x in (p, float(r["lat"]))])
cap_lon = F.create_map(*[F.lit(x) for p, r in centres.iterrows() for x in (p, float(r["lon"]))])
df = df.withColumn("dist_to_capital", haversine_km(F.col("geo_lat"), F.col("geo_lon"),
                                                   cap_lat[F.col("province")], cap_lon[F.col("province")]))

## 4. Khoảng cách tới thành phố gần nhất

Danh sách thành phố từ GeoNames `cities15000` (giấy phép CC BY 4.0): thành phố của Nga và các thành phố thuộc Crimea, Sevastopol (GeoNames gắn mã UA), bỏ các mục là quận nội thành (`PPLX`), địa danh lịch sử và bỏ hoang.

In [7]:
geo_cols = ["geonameid", "name", "asciiname", "alternatenames", "lat", "lon", "feature_class", "feature_code",
            "country", "cc2", "admin1", "admin2", "admin3", "admin4", "population", "elevation", "dem", "timezone", "modified"]
cities = (spark.read.csv("../data/raw/geonames/cities15000.txt", sep="\t", quote="",
                         schema=StructType([StructField(c, StringType()) for c in geo_cols]))
          .filter(~F.col("feature_code").isin("PPLX", "PPLH", "PPLQ"))
          .filter((F.col("country") == "RU") | ((F.col("country") == "UA") & F.col("admin1").isin("11", "20")))
          .select(F.col("asciiname").alias("city"), F.col("lat").cast("double").alias("city_lat"),
                  F.col("lon").cast("double").alias("city_lon")))
print(f"Số thành phố: {cities.count():,}")

Số thành phố: 995


Tính trên các tọa độ khác nhau (ghép chéo với danh sách thành phố, lấy khoảng cách nhỏ nhất) rồi ghép lại vào từng bản ghi.

In [8]:
coords = df.select("geo_lat", "geo_lon").distinct()
nearest = (coords.crossJoin(F.broadcast(cities))
                 .withColumn("d", haversine_km(F.col("geo_lat"), F.col("geo_lon"), F.col("city_lat"), F.col("city_lon")))
                 .groupBy("geo_lat", "geo_lon")
                 .agg(F.min(F.struct("d", "city")).alias("m"))
                 .select("geo_lat", "geo_lon", F.col("m.d").alias("dist_to_nearest_city"), F.col("m.city").alias("nearest_city")))
df = df.join(nearest, on=["geo_lat", "geo_lon"], how="left").cache()
df.select("dist_to_capital", "dist_to_nearest_city", "distance_to_most_frequent_coord_in_region").summary().show()

+-------+--------------------+--------------------+-----------------------------------------+
|summary|     dist_to_capital|dist_to_nearest_city|distance_to_most_frequent_coord_in_region|
+-------+--------------------+--------------------+-----------------------------------------+
|  count|             5459594|             5459594|                                  5459594|
|   mean|  28.590369541354494|   5.145434248433376|                       28.984930837862294|
| stddev|   64.74775378868928|   6.254083737175251|                        65.03222384330668|
|    min|0.009065941327430178| 9.16400226379929E-4|                                      0.0|
|    25%|   4.711039844126009|  2.3745282338184044|                       3.8360437224310897|
|    50%|   8.457791758134455|  4.2786272182682215|                         8.42458691078291|
|    75%|  18.831933570734282|   6.418664854974037|                        18.14325007852229|
|    max|  1831.3194611270092|  1236.7161663105855|         

## 5. Tương quan Spearman với giá

Giữa `price`, giá mỗi m² và ba đặc trưng khoảng cách (hai đặc trưng mới và đặc trưng cũ), trên toàn bộ dữ liệu và theo từng nhóm.

In [9]:
targets = ["price", "price_per_m2"]
features = ["dist_to_capital", "dist_to_nearest_city", "distance_to_most_frequent_coord_in_region"]
data = df.withColumn("price_per_m2", F.col("price") / F.col("area"))
vec = VectorAssembler(inputCols=targets + features, outputCol="v")

rows = {}
for name, cond in [("Toàn bộ", F.lit(True)), ("Moscow", F.col("price_group") == "Moscow"),
                   ("Saint Petersburg", F.col("price_group") == "Saint Petersburg"),
                   ("Phần còn lại", F.col("price_group") == "Phần còn lại")]:
    m = Correlation.corr(vec.transform(data.filter(cond)).select("v"), "v", "spearman").first()[0].toArray()
    for i, t in enumerate(targets):
        rows[(name, t)] = m[i, len(targets):]
spearman = pd.DataFrame(rows, index=features).round(3)
print(spearman.to_string())

                                          Toàn bộ              Moscow              Saint Petersburg              Phần còn lại             
                                            price price_per_m2  price price_per_m2            price price_per_m2        price price_per_m2
dist_to_capital                            -0.071       -0.034 -0.435       -0.600           -0.373       -0.403       -0.132       -0.105
dist_to_nearest_city                       -0.015        0.023 -0.016        0.004           -0.161       -0.146       -0.113       -0.084
distance_to_most_frequent_coord_in_region   0.014        0.053 -0.432       -0.597           -0.368       -0.397       -0.031        0.003
